# Tokenization

In [1]:
import re
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text = "The cats are playing."

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [2]:
enc = tokenizer(text, padding=True, truncation=True, return_tensors="pt", return_special_tokens_mask=True)
enc.keys()

KeysView({'input_ids': tensor([[ 101, 1996, 8870, 2024, 2652, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]]), 'special_tokens_mask': tensor([[1, 0, 0, 0, 0, 0, 1]])})

`input_ids` là chỉ số của từng token trong từ vựng, đầu vào của lớp embedding.

`attention_mask` bằng 1 tại token thật và 0 tại padding, để attention bỏ qua vị trí padding.

`special_tokens_mask` đánh dấu các token đặc biệt do tokenizer thêm vào.

In [3]:
ids = enc["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(ids)
special = enc["special_tokens_mask"][0].tolist()
table = pd.DataFrame({
    "position": range(len(ids)),
    "token": tokens,
    "id": ids,
    "special": special,
    "attention_mask": enc["attention_mask"][0].tolist(),
})
table

,position,token,id,special,attention_mask
0,0,[CLS],101,1,1
1,1,the,1996,0,1
2,2,cats,8870,0,1
3,3,are,2024,0,1
4,4,playing,2652,0,1
5,5,.,1012,0,1
6,6,[SEP],102,1,1


In [4]:
[t for t, s in zip(tokens, special) if s]

['[CLS]', '[SEP]']

In [5]:
content = [t for t, s in zip(tokens, special) if not s]
subwords = [t for t in content if t.startswith("##")]
counts = pd.DataFrame({
    "unit": ["words by whitespace", "words and punctuation", "tokens with special tokens", "tokens without special tokens", "subword continuation tokens"],
    "count": [len(text.split()), len(re.findall(r"\w+|[^\w\s]", text)), len(tokens), len(content), len(subwords)],
})
counts

,unit,count
0,words by whitespace,4
1,words and punctuation,5
2,tokens with special tokens,7
3,tokens without special tokens,5
4,subword continuation tokens,0


In [6]:
batch = tokenizer([text, "Cats play."], padding=True, return_tensors="pt")
batch["input_ids"], batch["attention_mask"]

(tensor([[ 101, 1996, 8870, 2024, 2652, 1012,  102],
         [ 101, 8870, 2377, 1012,  102,    0,    0]]),
 tensor([[1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 0, 0]]))